# Can we predict patient readmission?

This notebook builds an honest, reproducible baseline for predicting the `readmitted` column in `data.csv`.

**Primary estimand:** whether an eligible patient is readmitted at any time (`<30` or `>30`) versus `NO`, using information available by discharge. Set `TARGET_DEFINITION = "within_30_days"` below to study the stricter 30-day endpoint.

Key design choices:

- Split by `patient_nbr`, not by encounter, so one patient's history cannot appear in both development and test data.
- Keep the test set untouched until one model is selected on a validation set.
- Remove identifier columns from the predictors.
- Exclude terminal/hospice discharges by default: these patients are not meaningfully at risk of ordinary readmission.
- Compare against a no-skill prevalence baseline and report discrimination, calibration-oriented metrics, ranking lift, subgroup diagnostics, and uncertainty.

This is an exploratory prediction experiment, not a clinically validated model.

In [ ]:
from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, classification_report, ConfusionMatrixDisplay,
    PrecisionRecallDisplay, RocCurveDisplay, roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)


## 1. Load and audit the data

The common missing-value marker `?` is treated as missing. Mixed-type categorical fields are loaded consistently by disabling chunked type inference.

In [ ]:
DATA_PATH = Path("data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected {DATA_PATH.resolve()} to exist")

raw = pd.read_csv(DATA_PATH, na_values=["?"], low_memory=False)
print(f"Rows: {len(raw):,} | Columns: {raw.shape[1]:,} | Patients: {raw['patient_nbr'].nunique():,}")
display(raw.head())


In [ ]:
target_counts = raw["readmitted"].value_counts(dropna=False).rename("count").to_frame()
target_counts["percent"] = 100 * target_counts["count"] / len(raw)
display(target_counts)

audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing_n": raw.isna().sum(),
    "missing_pct": 100 * raw.isna().mean(),
    "unique_n": raw.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(audit.head(15).round(2))

repeat_rows = raw.duplicated("patient_nbr", keep=False).sum()
print(f"Rows from patients with multiple encounters: {repeat_rows:,} ({repeat_rows/len(raw):.1%})")
assert set(raw["readmitted"].dropna().unique()) <= {"NO", ">30", "<30"}


## 2. Define the outcome and eligible cohort

The original target is three-level. The question “whether a patient is readmitted” is interpreted as a binary endpoint:

- `any_readmission`: `<30` and `>30` are positive.
- `within_30_days`: only `<30` is positive; both `>30` and `NO` are negative.

Discharge disposition IDs 11, 19, 20, and 21 denote death/expired categories in this dataset's coding scheme; 13 and 14 denote hospice. Keeping these encounters makes `NO` partly deterministic rather than predictive. The filter is explicit and can be switched off for sensitivity analysis.

**Prediction time:** discharge. If the intended intervention happens at admission, post-admission fields (such as length of stay, procedures, discharge disposition, and medication changes) must also be removed.

In [ ]:
TARGET_DEFINITION = "any_readmission"  # or "within_30_days"
EXCLUDE_TERMINAL_OR_HOSPICE = True
TERMINAL_OR_HOSPICE_DISPOSITIONS = {11, 13, 14, 19, 20, 21}

df = raw.copy()
if EXCLUDE_TERMINAL_OR_HOSPICE:
    eligible = ~df["discharge_disposition_id"].isin(TERMINAL_OR_HOSPICE_DISPOSITIONS)
    print(f"Excluded {(~eligible).sum():,} terminal/hospice encounters ({(~eligible).mean():.1%}).")
    df = df.loc[eligible].copy()

if TARGET_DEFINITION == "any_readmission":
    df["target"] = df["readmitted"].isin(["<30", ">30"]).astype("int8")
elif TARGET_DEFINITION == "within_30_days":
    df["target"] = df["readmitted"].eq("<30").astype("int8")
else:
    raise ValueError("Unknown TARGET_DEFINITION")

print(f"Eligible encounters: {len(df):,}")
print(f"Positive outcome prevalence: {df['target'].mean():.1%}")
display(pd.crosstab(df["readmitted"], df["target"], margins=True))


## 3. Feature engineering

Encounter and patient identifiers are never predictors. The three high-cardinality diagnosis codes are converted to clinically broad ICD-9 families. Integer-coded administrative categories are explicitly treated as categorical rather than continuous quantities.

In [ ]:
def icd9_family(value):
    if pd.isna(value):
        return "Missing"
    code = str(value).strip().upper()
    if code.startswith("V"):
        return "Supplementary_V"
    if code.startswith("E"):
        return "External_E"
    try:
        number = float(code)
    except ValueError:
        return "Other"
    if 390 <= number < 460 or number == 785:
        return "Circulatory"
    if 460 <= number < 520 or number == 786:
        return "Respiratory"
    if 520 <= number < 580 or number == 787:
        return "Digestive"
    if 250 <= number < 251:
        return "Diabetes"
    if 800 <= number < 1000:
        return "Injury"
    if 710 <= number < 740:
        return "Musculoskeletal"
    if 580 <= number < 630 or number == 788:
        return "Genitourinary"
    if 140 <= number < 240:
        return "Neoplasms"
    return "Other"


def make_features(frame):
    X = frame.drop(columns=["readmitted", "target", "encounter_id", "patient_nbr"]).copy()
    for col in ["diag_1", "diag_2", "diag_3"]:
        X[f"{col}_family"] = X[col].map(icd9_family)
    X = X.drop(columns=["diag_1", "diag_2", "diag_3"])

    # These are category labels despite being stored as integers.
    categorical_ids = ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
    for col in categorical_ids:
        X[col] = X[col].astype("string")
    return X


X = make_features(df)
y = df["target"]
groups = df["patient_nbr"]

categorical_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_cols = X.columns.difference(categorical_cols).tolist()

print(f"Predictors: {X.shape[1]} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
display(X.head())


## 4. Patient-level train/validation/test split

The held-out test set represents previously unseen patients. A second group split within development data creates the validation set used for model selection. The printed table verifies disjoint patient sets and comparable outcome prevalence.

In [ ]:
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
dev_idx, test_idx = next(outer.split(X, y, groups))

inner = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE + 1)
train_rel, val_rel = next(inner.split(X.iloc[dev_idx], y.iloc[dev_idx], groups.iloc[dev_idx]))
train_idx, val_idx = dev_idx[train_rel], dev_idx[val_rel]

splits = {
    "train": (X.iloc[train_idx], y.iloc[train_idx], groups.iloc[train_idx]),
    "validation": (X.iloc[val_idx], y.iloc[val_idx], groups.iloc[val_idx]),
    "test": (X.iloc[test_idx], y.iloc[test_idx], groups.iloc[test_idx]),
}

summary = []
for name, (Xs, ys, gs) in splits.items():
    summary.append({"split": name, "encounters": len(Xs), "patients": gs.nunique(), "positive_rate": ys.mean()})
display(pd.DataFrame(summary).set_index("split").style.format({"positive_rate": "{:.2%}"}))

patient_sets = {name: set(part[2]) for name, part in splits.items()}
assert patient_sets["train"].isdisjoint(patient_sets["validation"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["validation"].isdisjoint(patient_sets["test"])


## 5. Compare candidate models on validation data

Two useful modeling families are compared with a prevalence-only baseline:

- Regularized logistic regression with one-hot encoded categories: interpretable, strong linear baseline.
- Histogram gradient boosting with ordinal-encoded categories: captures nonlinearities and interactions.

All imputation and encoding is inside each pipeline and is fitted only on training data. ROC AUC is the selection metric because the operational probability threshold is not yet specified. Average precision is also important, especially for the rarer 30-day endpoint.

In [ ]:
linear_preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True)),
        ("scale", StandardScaler(with_mean=False)),
    ]), numeric_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)),
    ]), categorical_cols),
])

ordinal_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median", add_indicator=True), numeric_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), categorical_cols),
])

candidates = {
    "prevalence_baseline": DummyClassifier(strategy="prior"),
    "logistic_regression": Pipeline([
        ("prep", linear_preprocessor),
        ("model", LogisticRegression(max_iter=500, solver="lbfgs", C=1.0)),
    ]),
    "hist_gradient_boosting": Pipeline([
        ("prep", ordinal_preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.07, max_iter=250, max_leaf_nodes=31,
            l2_regularization=1.0, early_stopping=True,
            random_state=RANDOM_STATE,
        )),
    ]),
}

X_train, y_train, _ = splits["train"]
X_val, y_val, _ = splits["validation"]

validation_rows = []
fitted_candidates = {}
for name, model in candidates.items():
    started = time.perf_counter()
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_val)[:, 1]
    prediction = (probability >= 0.5).astype(int)
    validation_rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_val, probability),
        "average_precision": average_precision_score(y_val, probability),
        "brier_score": brier_score_loss(y_val, probability),
        "balanced_accuracy@0.5": balanced_accuracy_score(y_val, prediction),
        "fit_seconds": time.perf_counter() - started,
    })
    fitted_candidates[name] = model

validation_results = pd.DataFrame(validation_rows).set_index("model").sort_values("roc_auc", ascending=False)
display(validation_results.style.format("{:.3f}"))


## 6. Final untouched-test evaluation

The best non-baseline model is selected by validation ROC AUC, cloned, and retrained on all development patients. Confidence intervals below use a patient-level bootstrap, preserving clustering from repeated encounters.

In [ ]:
eligible_models = validation_results.drop(index="prevalence_baseline")
best_name = eligible_models["roc_auc"].idxmax()
final_model = clone(candidates[best_name])

X_dev = pd.concat([splits["train"][0], splits["validation"][0]])
y_dev = pd.concat([splits["train"][1], splits["validation"][1]])
X_test, y_test, groups_test = splits["test"]

final_model.fit(X_dev, y_dev)
test_probability = final_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

test_metrics = pd.Series({
    "ROC AUC": roc_auc_score(y_test, test_probability),
    "Average precision": average_precision_score(y_test, test_probability),
    "Brier score (lower is better)": brier_score_loss(y_test, test_probability),
    "Accuracy at 0.5": accuracy_score(y_test, test_prediction),
    "Balanced accuracy at 0.5": balanced_accuracy_score(y_test, test_prediction),
}, name=best_name)
print(f"Selected model: {best_name}")
display(test_metrics.to_frame().style.format("{:.3f}"))
print(classification_report(y_test, test_prediction, target_names=["not readmitted", "readmitted"]))


In [ ]:
def patient_bootstrap_auc(y_true, probability, patient_ids, n_boot=500, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    work = pd.DataFrame({"y": np.asarray(y_true), "p": probability, "patient": np.asarray(patient_ids)})
    patient_to_rows = work.groupby("patient", sort=False).indices
    patients = np.array(list(patient_to_rows))
    scores = []
    for _ in range(n_boot):
        sampled = rng.choice(patients, size=len(patients), replace=True)
        rows = np.concatenate([patient_to_rows[p] for p in sampled])
        if work.iloc[rows]["y"].nunique() == 2:
            scores.append(roc_auc_score(work.iloc[rows]["y"], work.iloc[rows]["p"]))
    return np.percentile(scores, [2.5, 50, 97.5])


auc_ci = patient_bootstrap_auc(y_test, test_probability, groups_test)
print(f"Patient-bootstrap ROC AUC: {auc_ci[1]:.3f} (95% CI {auc_ci[0]:.3f}–{auc_ci[2]:.3f})")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[0], name=best_name)
axes[0].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[0].set_title("ROC curve")

PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[1], name=best_name)
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", linewidth=1, label="prevalence")
axes[1].set_title("Precision–recall curve")

ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, display_labels=["No", "Yes"], cmap="Blues", ax=axes[2]
)
axes[2].set_title("Confusion matrix at threshold 0.5")
plt.tight_layout()
plt.show()


### Risk-ranking usefulness

Threshold 0.5 is arbitrary. The table below sorts test encounters into ten equally sized risk groups. A useful ranking model should place substantially more outcomes in the highest-risk groups. “Lift” compares each group's observed outcome rate with overall prevalence.

In [ ]:
risk = pd.DataFrame({"outcome": y_test.to_numpy(), "probability": test_probability})
risk["risk_decile"] = pd.qcut(risk["probability"], 10, labels=False, duplicates="drop") + 1
lift = risk.groupby("risk_decile").agg(
    encounters=("outcome", "size"),
    observed_rate=("outcome", "mean"),
    mean_predicted_risk=("probability", "mean"),
).sort_index(ascending=False)
lift["lift_vs_average"] = lift["observed_rate"] / risk["outcome"].mean()
display(lift.style.format({
    "observed_rate": "{:.1%}", "mean_predicted_risk": "{:.1%}", "lift_vs_average": "{:.2f}×"
}))


### Subgroup diagnostic

These are descriptive checks, not a fairness certification. AUC can vary from sampling noise, differing case mix, and data quality. Small groups are omitted.

In [ ]:
test_context = df.iloc[test_idx][["race", "gender", "age"]].reset_index(drop=True).copy()
test_context["y"] = y_test.to_numpy()
test_context["p"] = test_probability

subgroup_rows = []
for attribute in ["race", "gender", "age"]:
    for level, part in test_context.groupby(attribute, dropna=False):
        if len(part) >= 200 and part["y"].nunique() == 2:
            subgroup_rows.append({
                "attribute": attribute,
                "group": str(level),
                "n": len(part),
                "prevalence": part["y"].mean(),
                "roc_auc": roc_auc_score(part["y"], part["p"]),
                "mean_prediction": part["p"].mean(),
            })
subgroups = pd.DataFrame(subgroup_rows).sort_values(["attribute", "n"], ascending=[True, False])
display(subgroups.style.format({"prevalence": "{:.1%}", "roc_auc": "{:.3f}", "mean_prediction": "{:.1%}"}))


## 7. Which input fields matter to this model?

Permutation importance measures the fall in test ROC AUC after shuffling one original field. It is computed on a fixed sample for speed. Correlated fields can share importance, and this is **predictive association, not causality**.

In [ ]:
importance_n = min(5000, len(X_test))
importance_sample = X_test.sample(importance_n, random_state=RANDOM_STATE)
importance_y = y_test.loc[importance_sample.index]

perm = permutation_importance(
    final_model, importance_sample, importance_y,
    scoring="roc_auc", n_repeats=3, random_state=RANDOM_STATE, n_jobs=1,
)
importance = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_sd": perm.importances_std,
}).sort_values("importance_mean", ascending=False).head(15)
display(importance.style.format({"importance_mean": "{:.4f}", "importance_sd": "{:.4f}"}))

plt.figure(figsize=(8, 5))
plot_data = importance.sort_values("importance_mean")
plt.barh(plot_data["feature"], plot_data["importance_mean"], xerr=plot_data["importance_sd"], alpha=0.8)
plt.xlabel("Decrease in ROC AUC when shuffled")
plt.title("Top permutation importances")
plt.tight_layout()
plt.show()


## 8. Answer and limitations

The cell below translates the held-out AUC into a deliberately restrained conclusion. In general, an AUC near 0.5 means no ranking ability; 0.6–0.7 is modest; 0.7–0.8 is useful discrimination in some settings but not sufficient by itself for deployment. Operational utility also depends on calibration, capacity, intervention benefit, harms, and the chosen threshold.

In [ ]:
auc = test_metrics["ROC AUC"]
if auc < 0.55:
    verdict = "The available variables provide little out-of-sample predictive signal."
elif auc < 0.70:
    verdict = "Yes, but only modestly: the model ranks risk better than chance, not strongly enough to use uncritically."
elif auc < 0.80:
    verdict = "Yes: the model shows potentially useful discrimination, subject to calibration and external validation."
else:
    verdict = "Yes, with strong discrimination in this held-out sample; leakage and external validity still require scrutiny."

top_rate = lift.iloc[0]["observed_rate"]
top_lift = lift.iloc[0]["lift_vs_average"]
print(verdict)
print(f"Held-out patient-level split ROC AUC: {auc:.3f} (95% CI {auc_ci[0]:.3f}–{auc_ci[2]:.3f}).")
print(f"The highest predicted-risk decile had {top_rate:.1%} observed outcomes ({top_lift:.2f}× the test-set average).")


### Limitations and next steps

1. **No external or temporal validation.** This estimates performance on unseen patients from the same dataset, not another hospital or future period.
2. **The split is not temporal.** If `encounter_id` encodes chronology, a documented date field would enable a better future-facing split.
3. **Prediction time matters.** This model is explicitly for discharge-time scoring. An admission-time model needs a narrower feature set.
4. **Repeated test encounters remain correlated.** The confidence interval resamples patients, but encounter-level metrics still weight frequent patients more heavily.
5. **Codes need semantic confirmation.** Administrative IDs and cohort exclusions should be verified against the source data dictionary.
6. **No causal interpretation.** Important variables identify association, not interventions that reduce readmission.
7. **Deployment needs calibration and utility analysis.** Select a threshold from clinical capacity/costs on validation data, assess calibration, audit equity, and validate prospectively.

For a stronger next experiment, compare this discharge-time model with an admission-time model, add a chronological holdout, tune only within group-aware cross-validation, and externally validate the locked pipeline.